# Buoc 1: Fix BestBuy Filter (Replace requests with Playwright)

**Goal:** `requests.get()` bi BestBuy block -> timeout 10/10.  
**Solution:** Dung Playwright, kiem tra sale + cao du lieu ngay trong 1 lan truy cap.  
**Return:** `List[Tuple[str, dict]]` giong Amazon.

## Cell 1: Reproduce bug - requests.get() bi block

In [ ]:
import requests
import time

test_urls = [
    "https://www.bestbuy.com/product/asus-zenbook-a14-14-fhd-oled-laptop-copilot-pc-snapdragon-x-plus-16gb-ram-512gb-ssd-zabriskie-beige/JJGGLH86J4",
    "https://www.bestbuy.com/product/asus-zenbook-14-14-fhd-oled-touch-screen-laptop-intel-core-ultra-7-16gb-ram-512gb-ssd-jasper-gray/JJGGLH7HXW",
]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

for url in test_urls:
    start = time.time()
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        elapsed = time.time() - start
        print(f"[{elapsed:.1f}s] Status: {resp.status_code} | Length: {len(resp.content)} | URL: {url[:70]}")
    except Exception as e:
        elapsed = time.time() - start
        print(f"[{elapsed:.1f}s] ERROR: {e} | URL: {url[:70]}")

[9.1s] ERROR: HTTPSConnectionPool(host='www.bestbuy.com', port=443): Read timed out. (read timeout=10) | URL: https://www.bestbuy.com/product/asus-zenbook-a14-14-fhd-oled-laptop-co
[9.2s] ERROR: HTTPSConnectionPool(host='www.bestbuy.com', port=443): Read timed out. (read timeout=10) | URL: https://www.bestbuy.com/product/asus-zenbook-14-14-fhd-oled-touch-scre


## Cell 2: Playwright - filter + scrape trong 1 lan truy cap

Thiet ke moi: mo trang 1 lan, kiem tra sale, neu co sale thi cao luon du lieu.  
Tiet kiem thoi gian vi khong can mo lai trang lan 2 de scrape.

In [2]:
import re
from playwright.async_api import async_playwright


async def filter_and_scrape_bestbuy_playwright(
    urls: list[str],
    headless: bool = False
) -> list[tuple[str, dict]]:
    """Filter BestBuy URLs for sale items and scrape data in one pass.

    Opens each URL once with Playwright. If the product is on sale,
    immediately scrapes title, brand, price, and features.

    Returns:
        List of (url, product_info) for products that are on sale.
        product_info keys: title, brand, sale_price, original_price,
                           savings, features
    """
    sale_items = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=headless,
            args=["--disable-blink-features=AutomationControlled", "--no-sandbox"]
        )
        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/121.0.0.0 Safari/537.36"
            ),
            viewport={"width": 1920, "height": 1080},
        )
        page = await context.new_page()

        for i, url in enumerate(urls, 1):
            print(f"[{i}/{len(urls)}] Opening: {url[:80]}...")
            try:
                await page.goto(url, timeout=60000, wait_until="domcontentloaded")
                await page.wait_for_timeout(2000)

                # --- Check sale indicators ---
                savings_elem = page.locator('[data-testid="price-block-total-savings-text"]')
                comp_value_elem = page.locator('[data-lu-target="comp_value"]')
                was_price_elem = page.locator('div.pricing-price__regular-price')

                has_savings = await savings_elem.count() > 0
                has_comp = await comp_value_elem.count() > 0
                has_was = await was_price_elem.count() > 0

                is_sale = has_savings or has_comp or has_was

                if not is_sale:
                    print(f"  -> Skip (not on sale)")
                    continue

                # --- On sale! Scrape immediately ---
                info = {}

                # Title
                title_el = page.locator("h1.h4")
                info["title"] = (
                    (await title_el.text_content()).strip()
                    if await title_el.count() > 0 else "Unknown"
                )

                # Brand
                brand_el = page.locator(
                    'div[data-component-name="ProductHeader"] a.c-button-link'
                )
                info["brand"] = (
                    (await brand_el.first.text_content()).strip()
                    if await brand_el.count() > 0 else None
                )

                # Sale price
                price_el = page.locator(
                    '[data-testid="price-block-customer-price"] span'
                )
                price_text = (
                    await price_el.first.text_content()
                    if await price_el.count() > 0 else "$0"
                )
                match = re.search(r'[\d,]+\.?\d*', price_text.replace(',', ''))
                info["sale_price"] = float(match.group()) if match else 0.0

                # Original / Was price
                orig_el = page.locator('[data-testid="price-block-regular-price"] span')
                if await orig_el.count() > 0:
                    orig_text = await orig_el.first.text_content()
                    orig_match = re.search(r'[\d,]+\.?\d*', orig_text.replace(',', ''))
                    info["original_price"] = float(orig_match.group()) if orig_match else None
                else:
                    info["original_price"] = None

                # Savings text
                if has_savings:
                    info["savings"] = (await savings_elem.first.text_content()).strip()
                else:
                    info["savings"] = None

                # Features (click button to expand)
                features = ""
                features_btn = page.locator('button:has(h3:text("Features"))')
                if await features_btn.count() > 0:
                    await features_btn.first.click()
                    try:
                        await page.locator('[data-testid="brix-sheet-content"]').wait_for(timeout=5000)
                        feat_el = page.locator('[data-testid="brix-sheet-content"]')
                        features = (await feat_el.first.text_content()) or ""
                    except Exception:
                        pass
                    await page.keyboard.press("Escape")
                    await page.wait_for_timeout(500)
                info["features"] = features[:1500]

                sale_items.append((url, info))
                print(
                    f"  -> SALE! ${info['sale_price']:.2f}"
                    f" (was {info.get('original_price', '?')})"
                    f" | {info.get('savings', '')}"
                    f" | {info['title'][:60]}"
                )

            except Exception as e:
                print(f"  -> ERROR: {e}")
                continue

        await browser.close()

    print(f"\nResult: {len(sale_items)}/{len(urls)} products are on sale")
    return sale_items

## Cell 3: Test voi 2 URLs

In [3]:
import time

test_urls = [
    "https://www.bestbuy.com/product/asus-zenbook-a14-14-fhd-oled-laptop-copilot-pc-snapdragon-x-plus-16gb-ram-512gb-ssd-zabriskie-beige/JJGGLH86J4",
    "https://www.bestbuy.com/product/asus-zenbook-14-14-fhd-oled-touch-screen-laptop-intel-core-ultra-7-16gb-ram-512gb-ssd-jasper-gray/JJGGLH7HXW",
]

start = time.time()
results = await filter_and_scrape_bestbuy_playwright(test_urls, headless=False)
elapsed = time.time() - start

print(f"\n--- Total time: {elapsed:.1f}s ---")
print(f"--- Sale items: {len(results)}/{len(test_urls)} ---")

[1/2] Opening: https://www.bestbuy.com/product/asus-zenbook-a14-14-fhd-oled-laptop-copilot-pc-s...
  -> ERROR: Page.goto: net::ERR_HTTP2_PROTOCOL_ERROR at https://www.bestbuy.com/product/asus-zenbook-a14-14-fhd-oled-laptop-copilot-pc-snapdragon-x-plus-16gb-ram-512gb-ssd-zabriskie-beige/JJGGLH86J4
Call log:
  - navigating to "https://www.bestbuy.com/product/asus-zenbook-a14-14-fhd-oled-laptop-copilot-pc-snapdragon-x-plus-16gb-ram-512gb-ssd-zabriskie-beige/JJGGLH86J4", waiting until "domcontentloaded"

[2/2] Opening: https://www.bestbuy.com/product/asus-zenbook-14-14-fhd-oled-touch-screen-laptop-...
  -> ERROR: Page.goto: net::ERR_HTTP2_PROTOCOL_ERROR at https://www.bestbuy.com/product/asus-zenbook-14-14-fhd-oled-touch-screen-laptop-intel-core-ultra-7-16gb-ram-512gb-ssd-jasper-gray/JJGGLH7HXW
Call log:
  - navigating to "https://www.bestbuy.com/product/asus-zenbook-14-14-fhd-oled-touch-screen-laptop-intel-core-ultra-7-16gb-ram-512gb-ssd-jasper-gray/JJGGLH7HXW", waiting until "domcontentl

## Cell 4: Xem chi tiet ket qua

In [4]:
for url, info in results:
    print("=" * 80)
    print(f"URL:      {url}")
    print(f"Title:    {info.get('title', 'N/A')}")
    print(f"Brand:    {info.get('brand', 'N/A')}")
    print(f"Sale $:   ${info.get('sale_price', 0):.2f}")
    print(f"Was $:    {info.get('original_price', 'N/A')}")
    print(f"Savings:  {info.get('savings', 'N/A')}")
    print(f"Features: {info.get('features', '')[:200]}...")
    print()